# 04 — Selected-dataset diagnostic experiments
Run All: shared VDD → validation screen → longer/finer training → freeze → final test metrics → test videos. No execution toggles or pasted run directories.

Despite its retained filename, this is an exploratory single-seed diagnostic, **not the controlled AAD paper protocol**. The original `aad_experiment_workflow.ipynb`, AAD configs and CLIs remain separate and unchanged. Do not tune models from test results.

In [1]:
from pathlib import Path
import subprocess
import sys
import importlib

REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    update = subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, capture_output=True, text=True)
    print(update.stdout, flush=True)
    if update.returncode:
        raise RuntimeError('Cannot safely update the Colab checkout. Save/commit conflicting local edits, resolve the Git error, then rerun setup. No local edits were discarded.\n' + update.stderr)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
required_helpers = ('notebook_data.py', 'notebook_display.py', 'notebook_workflows.py')
missing_helpers = [name for name in required_helpers if not (IMPLEMENTATION_ROOT / 'src' / name).is_file()]
revision = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print(f'Code checkout: {REPO_ROOT} @ {revision}', flush=True)
if missing_helpers:
    raise RuntimeError(f'Colab checkout is missing repository files: {missing_helpers}. These are NOT pip packages. Commit and push the complete notebook/source update to master, then rerun this setup cell. A kernel restart alone cannot download unpublished files.')
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
sys.path[:] = [str(IMPLEMENTATION_ROOT)] + [entry for entry in sys.path if entry != str(IMPLEMENTATION_ROOT)]
for name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[name]
importlib.invalidate_caches()
from src.notebook_utils import validate_runtime
print(validate_runtime(require_cuda=True))
from src.notebook_data import prepare_data
prepared = prepare_data(IMPLEMENTATION_ROOT)


Already up to date.

Code checkout: /content/phd_research @ 70e726d


RuntimeError: Colab checkout is missing repository files: ['notebook_data.py', 'notebook_display.py', 'notebook_workflows.py']. These are NOT pip packages. Commit and push the complete notebook/source update to master, then rerun this setup cell. A kernel restart alone cannot download unpublished files.

## Declared candidate budgets
Three architectures, eight epochs each at 32×32; same split, seed and input budget. Rank on validation macro-F1, accuracy, parameter count, then name. Checkpoints within each run use validation loss.

In [ ]:
from src.notebook_workflows import show_experiment_plan, run_screen, train_winner, final_evaluate
screen_dir = final_data = None  # Clear any previous run before starting.
plan = show_experiment_plan(prepared)


In [ ]:
screen_dir = run_screen(prepared)


## Longer training at finer resolution
Retrain the validation-selected architecture from scratch for 24 epochs at 64×64, then freeze its lowest-validation-loss checkpoint. A training stage, not a controlled resolution ablation.

In [ ]:
final_data = train_winner(prepared, screen_dir)


## Final test metrics, then playable test predictions
Run All intentionally opens test only after the complete screen and compatible longer training. Full test metrics come before up to five test videos/probabilities. Rerunning this last cell reuses its saved report. Do not tune or rerank from test feedback.

In [ ]:
final_report = final_evaluate(final_data, screen_dir)
